# 10 임베딩 기반 시계열 분석 (Phase 2)

Phase1 Best 알고리즘 × **6 임베딩** (PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST)

- 오차지표: MAE, RMSE, MAPE, MASE → 조건별 최적 임베딩 조합 선정

### ⓪ 환경 설정

In [ ]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())

phase1_best = pd.read_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv')


### ① Phase 2 하이브리드 실험

In [ ]:
p2_cache = DATA_PROCESSED / 'phase2_results.parquet'
if p2_cache.exists():
    phase2 = pd.read_parquet(p2_cache)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    print('Phase2 캐시 로드 |', len(phase2))
else:
    emb_cache = build_global_embedding_cache(df)
    p2_sbc = run_phase2_all(df, feat_df, 'SBC_CLUSTER', 'SBC', phase1_best, emb_cache=emb_cache)
    p2_ml = run_phase2_all(df, feat_df, 'ML_CLUSTER', 'ML', phase1_best, emb_cache=emb_cache)
    phase2 = pd.concat([p2_sbc, p2_ml], ignore_index=True)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    phase2.to_parquet(p2_cache, index=False)
    phase2_summary.to_csv(DATA_PROCESSED / 'phase2_summary.csv', index=False)
    phase2_best.to_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv', index=False)

display(phase2_best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== Best 임베딩 빈도 ===')
print(phase2_best['embedding'].value_counts())

final_best = pick_final_per_condition(phase1_best, phase2_best)
final_best.to_csv(DATA_PROCESSED / 'final_best_per_condition.csv', index=False)
print('Phase1 vs Phase2 최종 | Phase2 승률:', round((final_best.winner=='Phase2').mean(), 3))
